1. Import things

In [1]:
import requests
import folium
import geopandas as gpd
from shapely.geometry import shape
from pyproj import CRS
import json
import os

2. Import API

In [2]:
import requests
import geopandas as gpd
def load_data():
 mobil_fleet = 'https://geoserver.mapid.io/layers_new/get_layer?api_key=70b6758f25e94e6a849a32d935bce369&layer_id=69885cbe2d33abfe640dff7e&project_id=697e526a949d92a51f0b4816'
 pelanggan = 'https://geoserver.mapid.io/layers_new/get_layer?api_key=70b6758f25e94e6a849a32d935bce369&layer_id=69885de72d33abfe640e60c8&project_id=697e526a949d92a51f0b4816'
 area = 'https://geoserver.mapid.io/layers_new/get_layer?api_key=70b6758f25e94e6a849a32d935bce369&layer_id=69885e69bbaa47f3ce175cde&project_id=697e526a949d92a51f0b4816'
 response_mobil = requests.get(mobil_fleet)
 response_pelanggan = requests.get(pelanggan)
 response_area = requests.get(area)
 if response_mobil.status_code == 200 and response_pelanggan.status_code == 200 and response_area.status_code == 200:
   geojson_mobil = response_mobil.json()
   geojson_pelanggan = response_pelanggan.json()
   geojson_area = response_area.json()
    #ubah ke GeoDataFrame
   gdf_mobil = gpd.GeoDataFrame.from_features(geojson_mobil['features'])
   gdf_pelanggan = gpd.GeoDataFrame.from_features(geojson_pelanggan['features'])
   gdf_area = gpd.GeoDataFrame.from_features(geojson_area['features'])
    # Tetapkan CRS
   gdf_mobil.crs = "epsg:4326"
   gdf_pelanggan.crs = "epsg:4326"
   gdf_area.crs = "epsg:4326"
   return gdf_mobil, gdf_pelanggan, gdf_area
 else:
   print("Failed to fetch data")
   return None, None, None
gdf_mobil, gdf_pelanggan, gdf_area = load_data()

3. Filter mobil aktif dan kapasitas sesuai

In [3]:
def filter_mobil_aktif(gdf_mobil, jumlah_penumpang):
 return gdf_mobil[(gdf_mobil['STATUS'] == 'AKTIF') & (gdf_mobil['KAPASITAS'] >= jumlah_penumpang)].copy()

jumlah_penumpang = gdf_pelanggan.iloc[0]['JUMLAH_PENUMPANG']
gdf_mobil_aktif = filter_mobil_aktif(gdf_mobil, jumlah_penumpang)

4. Hitung jarak ke mobil

In [4]:
def hitung_jarak(gdf_mobil_aktif, pelanggan_point):
 gdf_mobil_aktif['JARAK'] = gdf_mobil_aktif.geometry.distance(pelanggan_point)
 return gdf_mobil_aktif
pelanggan_point = gdf_pelanggan.geometry.iloc[0]
gdf_mobil_aktif = hitung_jarak(gdf_mobil_aktif, pelanggan_point)



C:\Users\ASUS\AppData\Local\Temp\ipykernel_24940\4272011980.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'distance' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_mobil_aktif['JARAK'] = gdf_mobil_aktif.geometry.distance(pelanggan_point)


5. Dapatkan info area mobil & pelanggan

In [5]:
def spatial_join_area(gdf_obj, gdf_area):
 return gpd.sjoin(gdf_obj, gdf_area, how='left', predicate='within')
mobil_area = spatial_join_area(gdf_mobil_aktif, gdf_area)
pelanggan_area = spatial_join_area(gdf_pelanggan, gdf_area)

6. Skoring multi kriteria mobil

In [6]:
import math
def hitung_jarak_km(lat1, lon1, lat2, lon2):
 # Radius bumi dalam km
 R = 6371
 dlat = math.radians(lat2 - lat1)
 dlon = math.radians(lon2 - lon1)
 a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1))*math.cos(math.radians(lat2))*math.sin(dlon/2)**2
 c = 2*math.atan2(math.sqrt(a), math.sqrt(1-a))
 jarak = R * c
 return jarak
def skoring_mobil(row, pelanggan):
 skor = 0
 
# Jarak (semakin dekat semakin tinggi)
 jarak_km = hitung_jarak_km(row.geometry.y, row.geometry.x, pelanggan.geometry.y, pelanggan.geometry.x)
 if jarak_km < 1:
   skor += 100
 elif jarak_km < 2:
   skor += 50
 elif jarak_km < 3:
   skor += 25
 else:
   skor += 0
 # Area tidak hujan/banjir
 skor += 10 if str(row.get("HUJAN", "")).upper() == 'TIDAK' else 0
 skor += 10 if str(row.get("BANJIR", "")).upper() == 'TIDAK' else 0
 # Kepadatan rendah lebih baik
 skor += 10 if row['KEPADATAN'] == 'RENDAH' else 5 if row['KEPADATAN'] == 'SEDANG' else 0
 # Traffic lancar lebih baik
 skor += 10 if row['TRAFFIC'] == 'LANCAR' else 5 if row['TRAFFIC'] == 'SEDANG' else 0
 # Rating driver
 skor += 10 if row['RATING'] >= 4.8 else 5
 return skor

7. Mobil terbaik berdasarkan skor dan jarak

In [7]:
def pilih_mobil_terbaik(mobil_area, pelanggan):
 # Hitung skor untuk setiap mobil
 mobil_area['SKOR'] = mobil_area.apply(lambda row:
skoring_mobil(row, pelanggan), axis=1)

 # Hitung jarak dalam km
 mobil_area['JARAK_KM'] = mobil_area.apply(lambda row:
hitung_jarak_km(row.geometry.y, row.geometry.x, pelanggan.geometry.y,
pelanggan.geometry.x), axis=1)

 # Urutkan berdasarkan JARAK_KM (asc), lalu SKOR (desc)
 mobil_area_sorted = mobil_area.sort_values(['JARAK_KM', 'SKOR'],
ascending=[True, False])
 return mobil_area_sorted
pelanggan = gdf_pelanggan.iloc[0]
mobil_area_sorted = pilih_mobil_terbaik(mobil_area, pelanggan)
# Tampilkan mobil terpilih
mobil_terpilih = mobil_area_sorted.iloc[0]
print("**Mobil Terpilih:**")
print(mobil_terpilih[['NAMA_DRIVER', 'TIPE', 'KAPASITAS', 'RATING',
'JARAK_KM', 'SKOR']])
# Tampilkan mobil tidak terpilih
mobil_tidak_terpilih = mobil_area_sorted.iloc[1:]
# Mengabaikan baris pertama (mobil terpilih)
print("\n**Mobil Tidak Terpilih:**")
print(mobil_tidak_terpilih[['NAMA_DRIVER', 'TIPE', 'KAPASITAS',
'RATING', 'JARAK_KM', 'SKOR']])

**Mobil Terpilih:**
NAMA_DRIVER        Siti
TIPE              Sedan
KAPASITAS             4
RATING              4.9
JARAK_KM       0.539754
SKOR                120
Name: 3, dtype: object

**Mobil Tidak Terpilih:**
  NAMA_DRIVER       TIPE  KAPASITAS  RATING  JARAK_KM  SKOR
4        Budi        MPV          6     4.8  0.673906   110
1        Dewi  Hatchback          4     4.6  0.929222   145
0        Bima        SUV          4     4.5  2.210008    30


8. 

In [8]:
import json
# Buat GeoJSON mobil terpilih
mobil_terpilih_geojson = {
 "type": "Feature",
 "properties": {
 "NAMA_DRIVER": mobil_terpilih['NAMA_DRIVER'],
 "TIPE": mobil_terpilih['TIPE'],
 "KAPASITAS": int(mobil_terpilih['KAPASITAS']),
 "RATING": float(mobil_terpilih['RATING']),
 "JARAK_KM": float(mobil_terpilih['JARAK_KM']),
 "SKOR": int(mobil_terpilih['SKOR'])
 },
 "geometry": {
 "type": "Point",
 "coordinates": [mobil_terpilih.geometry.x,
mobil_terpilih.geometry.y]
 }
}
# Simpan GeoJSON ke file
output_dir = "."
output_path = os.path.join(output_dir, "mobil_terpilih.geojson")

with open(output_path, 'w') as f:
 json.dump(mobil_terpilih_geojson, f, indent=2)

9. 